In [6]:
%pip install openpyxl --timeout 120

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import openpyxl

print("openpyxl version:", openpyxl.__version__)

df = pd.read_excel("../data/raw/2011.xlsx")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

openpyxl version: 3.1.5
Dataset loaded successfully!
Rows: 39820
Columns: 35


,Application Number,Title,Application Date,Status,Publication Number,Publication Date(U/S 11A),Publication Type,Application Filing Date,Priority Number,Priority Country,...,PARENT APPLICATION NUMBER,PARENT APPLICATION FILING DATE,REQUEST FOR EXAMINATION DATE,FIRST EXAMINATION REPORT DATE,Date Of Certificate Issue,POST GRANT JOURNAL DATE,REPLY TO FER DATE,PCT INTERNATIONAL APPLICATION NUMBER,PCT INTERNATIONAL FILING DATE,Application Status
0,1/MUM/2011,WATER LIFTING SYSTEM,01/01/2011,Published,26/2013,28/06/2013,INA,01/01/2011,NaN,NaN,...,NaN,NaN,--,NaN,NaN,NaN,NaN,NaN,31/12/2010,Deemed to be withdrawn u/s 11B(4)
1,1/KOLNP/2011,CONTENT PERSONALIZATION USING FILE REPAIR REQU...,01/01/2011,Published,37/2016,02/09/2016,INA,01/01/2011,NaN,NaN,...,NaN,NaN,22/05/2012,NaN,NaN,NaN,08/10/2018,PCT/EP2008/004445,04/06/2008,Application Refused U/S 15
2,2/KOLNP/2011,METHOD FOR TREATING FLOOR POLISH STRIPPING WAS...,01/01/2011,Published,37/2016,02/09/2016,INA,01/01/2011,Japan2008-176353,Japan,...,NaN,NaN,27/06/2012,NaN,NaN,NaN,NaN,PCT/IB2009/006558,23/06/2009,Abandoned U/s 21(1)
3,2395/MUMNP/2012,CAMERA GUARANTEEING AUTHENTICITY OF VIDEOS OR ...,03/01/2011,Published,02/2014,10/01/2014,INA,03/01/2011,U.S.A.61/059769,U.S.A.,...,NaN,NaN,03/01/2011,NaN,NaN,NaN,02/01/2017,PCT/IB2009/052392,05/06/2009,Abandoned U/s 21(1)
4,49/DELNP/2011,A SYSTEM AND METHOD FOR COSMETIC TREATMENT AND...,03/01/2011,Published,34/2013,23/08/2013,INA,03/01/2011,"U.S.A.61/059,477",U.S.A.,...,NaN,NaN,30/05/2012,19/03/2018,20/03/2020,03/04/2020,17/12/2018,PCT/US2009/046475,05/06/2009,"Granted Application, Patent Number :335462"


In [8]:
print("Available columns:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Available columns:

1. Application Number
2. Title
3. Application Date
4. Status
5. Publication Number
6. Publication Date(U/S 11A)
7. Publication Type
8. Application Filing Date
9. Priority Number
10. Priority Country
11. Priority Date
12. Field Of Invention
13. Classification (IPC)
14. Inventor Name
15. Inventor Address
16. Inventor Country
17. Inventor Nationality
18. Applicant Name
19. Applicant Address
20. Applicant Country
21. Applicant Nationality
22. Application Type
23. E-MAIL (As Per Record)
24. ADDITIONAL-EMAIL (As Per Record)
25. E-MAIL (UPDATED Online)
26. PARENT APPLICATION NUMBER
27. PARENT APPLICATION FILING DATE
28. REQUEST FOR EXAMINATION DATE
29. FIRST EXAMINATION REPORT DATE
30. Date Of Certificate Issue
31. POST GRANT JOURNAL DATE
32. REPLY TO FER DATE
33. PCT INTERNATIONAL APPLICATION NUMBER
34. PCT INTERNATIONAL FILING DATE
35. Application Status


In [9]:

# Select useful columns for our initial RAG pipeline

selected_columns = [
    "Application Number",
    "Title",
    "Field Of Invention",
    "Classification (IPC)",
    "Inventor Name",
    "Applicant Name",
    "Status",
    "Application Filing Date",
    "Publication Number"
]

patent_df = df[selected_columns].copy()

print("Original dataset shape:", df.shape)
print("Selected dataset shape:", patent_df.shape)

display(patent_df.head())

Original dataset shape: (39820, 35)
Selected dataset shape: (39820, 9)


,Application Number,Title,Field Of Invention,Classification (IPC),Inventor Name,Applicant Name,Status,Application Filing Date,Publication Number
0,1/MUM/2011,WATER LIFTING SYSTEM,MECHANICAL ENGINEERING,F04F3/00; F04F1/00,#Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...,#Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...,Published,01/01/2011,26/2013
1,1/KOLNP/2011,CONTENT PERSONALIZATION USING FILE REPAIR REQU...,NaN,"H04L 12/18,G06Q 30/00",#PERKUHN Heiko#REMBARZ Rene#VORWERK Marc#WI...,#TELEFONAKTIEBOLAGET LM ERICSSON (publ),Published,01/01/2011,37/2016
2,2/KOLNP/2011,METHOD FOR TREATING FLOOR POLISH STRIPPING WAS...,NaN,C02F 1/00,#KAYAMORI Satoshi#KONDO Tetsuro#KOMODA Susumu,#DIVERSEY INC.,Published,01/01/2011,37/2016
3,2395/MUMNP/2012,CAMERA GUARANTEEING AUTHENTICITY OF VIDEOS OR ...,PHYSICS,"G03B19/00,G03B43/00,G03B17/00",#CHOUDURY Spandan,#CHOUDURY Spandan,Published,03/01/2011,02/2014
4,49/DELNP/2011,A SYSTEM AND METHOD FOR COSMETIC TREATMENT AND...,MECHANICAL ENGINEERING,A61B 8/00,"#BARTHE, PETER, G.#SLAYTON, MICHAEL, H.#MAKIN,...","#ULTHERA, INC.",Published,03/01/2011,34/2013


In [10]:
print(patent_df.iloc[0].to_string())

Application Number                                                1/MUM/2011
Title                                                   WATER LIFTING SYSTEM
Field Of Invention                                    MECHANICAL ENGINEERING
Classification (IPC)                                      F04F3/00; F04F1/00
Inventor Name              #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...
Applicant Name             #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...
Status                                                             Published
Application Filing Date                                           01/01/2011
Publication Number                                                   26/2013


In [11]:
print("Missing values in each column:\n")
print(patent_df.isnull().sum())

Missing values in each column:

Application Number           0
Title                        0
Field Of Invention         320
Classification (IPC)        19
Inventor Name              167
Applicant Name              51
Status                       0
Application Filing Date      0
Publication Number           0
dtype: int64


In [12]:
# Create a clean copy without modifying the original dataset
clean_df = patent_df.copy()

# Replace missing values in text columns with empty strings
text_columns = [
    "Title",
    "Field Of Invention",
    "Classification (IPC)",
    "Inventor Name",
    "Applicant Name",
    "Status",
    "Application Number",
    "Publication Number"
]

clean_df[text_columns] = clean_df[text_columns].fillna("")

# Check missing values after cleaning
print("Missing values after cleaning:\n")
print(clean_df.isnull().sum())

print("\nCleaned dataset shape:", clean_df.shape)

Missing values after cleaning:

Application Number         0
Title                      0
Field Of Invention         0
Classification (IPC)       0
Inventor Name              0
Applicant Name             0
Status                     0
Application Filing Date    0
Publication Number         0
dtype: int64

Cleaned dataset shape: (39820, 9)


In [13]:
# Count duplicate application numbers
duplicate_count = clean_df["Application Number"].duplicated().sum()

print("Total records:", len(clean_df))
print("Unique application numbers:",
      clean_df["Application Number"].nunique())
print("Duplicate application numbers:", duplicate_count)

Total records: 39820
Unique application numbers: 38649
Duplicate application numbers: 1171


In [14]:
# Find all rows with duplicated application numbers
duplicate_rows = clean_df[
    clean_df["Application Number"].duplicated(keep=False)
].sort_values("Application Number")

print("Rows belonging to duplicated application numbers:",
      len(duplicate_rows))

display(duplicate_rows.head(10))

Rows belonging to duplicated application numbers: 2329


,Application Number,Title,Field Of Invention,Classification (IPC),Inventor Name,Applicant Name,Status,Application Filing Date,Publication Number
3874,1000/DELNP/2011,"""RETRANSMISSION RESOURCE ALLOCATION METHOD FOR...",NO SUBJECT,H04B 7/26,#SOENG HUN KIM#GERT-JAN VAN LIESHOUT,"#SAMSUNG ELECTRONICS CO., LTD.",Published,10/02/2011,48/2011
3869,1000/DELNP/2011,"""RETRANSMISSION RESOURCE ALLOCATION METHOD FOR...",NO SUBJECT,H04B 7/26,#SOENG HUN KIM#GERT-JAN VAN LIESHOUT,"#SAMSUNG ELECTRONICS CO., LTD.",Published,10/02/2011,48/2011
22483,1000/KOL/2011,CONTROL PANEL FOR A MEASURING DEVICE,MECHANICAL ENGINEERING,G06F3/044,#HELMUT BROCKHAUS,#KROHNE MESSTECHNIK GMBH,Published,27/07/2011,43/2014
22488,1000/KOL/2011,CONTROL PANEL FOR A MEASURING DEVICE,MECHANICAL ENGINEERING,G06F3/044,#HELMUT BROCKHAUS,#KROHNE MESSTECHNIK GMBH,Published,27/07/2011,43/2014
38089,10020/DELNP/2011,SOLVER-BASED VISUALIZATION FRAMEWORK,COMPUTER SCIENCE,G06F 17/50,"#BECKMAN, BRIAN, C.#GREEN, DAVID G.#MITAL, VIJ...",#MICROSOFT CORPORATION,Published,19/12/2011,42/2012
38099,10020/DELNP/2011,SOLVER-BASED VISUALIZATION FRAMEWORK,COMPUTER SCIENCE,G06F 17/50,"#BECKMAN, BRIAN, C.#GREEN, DAVID G.#MITAL, VIJ...",#MICROSOFT CORPORATION,Published,19/12/2011,42/2012
38086,10021/DELNP/2011,SEARCHING REGULAR EXPRESSIONS WITH VIRTUALIZED...,COMPUTER SCIENCE,G06F 17/00,"#EGURO, KENNETH H.#FORIN, ALESSANDRO",#MICROSOFT CORPORATION,Published,19/12/2011,42/2012
38088,10021/DELNP/2011,SEARCHING REGULAR EXPRESSIONS WITH VIRTUALIZED...,COMPUTER SCIENCE,G06F 17/00,"#EGURO, KENNETH H.#FORIN, ALESSANDRO",#MICROSOFT CORPORATION,Published,19/12/2011,42/2012
38098,10021/DELNP/2011,SEARCHING REGULAR EXPRESSIONS WITH VIRTUALIZED...,COMPUTER SCIENCE,G06F 17/00,"#EGURO, KENNETH H.#FORIN, ALESSANDRO",#MICROSOFT CORPORATION,Published,19/12/2011,42/2012
38085,10022/DELNP/2011,PERSISTENT MEDIA PLAYBACK,COMPUTER SCIENCE,G06Q 50/00,"#GIBSON, CHAD, C.#TAHIR, SARA#EVANS, CHRISTOPH...",#MICROSOFT CORPORATION,Published,19/12/2011,45/2012


In [15]:
# Check duplicates across all 9 selected columns
exact_duplicate_count = clean_df.duplicated(
    subset=clean_df.columns.tolist()
).sum()

print("Exact duplicate rows:", exact_duplicate_count)

# Count unique application numbers after removing exact duplicate rows
deduplicated_df = clean_df.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(deduplicated_df))
print(
    "Unique application numbers:",
    deduplicated_df["Application Number"].nunique()
)

Exact duplicate rows: 1094
Rows after removing exact duplicates: 38726
Unique application numbers: 38649


In [16]:

def create_patent_document(row):
    return (
        f"Application Number: {row['Application Number']}\n"
        f"Title: {row['Title']}\n"
        f"Field of Invention: {row['Field Of Invention']}\n"
        f"IPC Classification: {row['Classification (IPC)']}\n"
        f"Inventor: {row['Inventor Name']}\n"
        f"Applicant: {row['Applicant Name']}\n"
        f"Status: {row['Status']}\n"
        f"Application Filing Date: {row['Application Filing Date']}\n"
        f"Publication Number: {row['Publication Number']}"
    )

# Create one text document for each remaining row
patent_documents = deduplicated_df.apply(
    create_patent_document, axis=1
).tolist()

print("Total patent documents:", len(patent_documents))
print("\nSample patent document:\n")
print(patent_documents[0])

Total patent documents: 38726

Sample patent document:

Application Number: 1/MUM/2011
Title: WATER LIFTING SYSTEM
Field of Invention: MECHANICAL ENGINEERING
IPC Classification: F04F3/00; F04F1/00
Inventor: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Applicant: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Status: Published
Application Filing Date: 01/01/2011
Publication Number: 26/2013


In [17]:

# Check document quality before embedding

empty_documents = sum(
    1 for doc in patent_documents if not doc.strip()
)

document_lengths = [
    len(doc) for doc in patent_documents
]

print("Total documents:", len(patent_documents))
print("Empty documents:", empty_documents)
print("Shortest document length:", min(document_lengths))
print("Longest document length:", max(document_lengths))
print(
    "Average document length:",
    round(sum(document_lengths) / len(document_lengths), 2)
)

Total documents: 38726
Empty documents: 0
Shortest document length: 202
Longest document length: 1915
Average document length: 354.78


In [18]:
%pip install sentence-transformers faiss-cpu

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.3 MB 2.4 MB/s eta 0:00:07
   -- ------------------------------------- 1.0/16.3 MB 2.4 MB/s eta 0:00:07
   --- ------------------------------------ 1.3/16.3 MB 2.4 MB/s eta 0:00:07
   ---- ----------------------------------- 1.8/16.3 MB 2.4 MB/s eta 0:00:07
   ----- ---------------------------------- 2.4/16.3 MB 2.4 MB/s eta 0:00:06
   ------- -------------------------------- 2.9/16.3 MB 2.4 MB/s eta 0:00:06
   -------- ----------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Embedding model loaded successfully!")
print("Embedding dimensions:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\navee\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\navee\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Embedding dimensions: 384


C:\Users\navee\AppData\Local\Temp\ipykernel_4040\4166842181.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model.get_sentence_embedding_dimension())


In [1]:

from pathlib import Path
import numpy as np

# 1. Create the output folder
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

embedding_file = output_dir / "patent_embeddings.npy"
checkpoint_file = output_dir / "patent_embeddings_progress.txt"

# 2. Configuration
total = len(patent_documents)
dimension = model.get_embedding_dimension()
batch_size = 32

print("Total documents:", total)
print("Embedding dimensions:", dimension)

# 3. Resume existing work or initialize new files
if embedding_file.exists() and checkpoint_file.exists():
    embeddings = np.load(embedding_file, mmap_mode="r+")
    
    if embeddings.shape != (total, dimension):
        raise ValueError(
            "Embedding file shape does not match the current documents. "
            "Check the files before continuing."
        )

    start = int(checkpoint_file.read_text().strip())

    if not 0 <= start <= total or (start != total and start % batch_size != 0):
        raise ValueError("Invalid checkpoint. Check the progress file.")

    print(f"Resuming from document {start}/{total}")

elif not embedding_file.exists() and not checkpoint_file.exists():
    embeddings = np.lib.format.open_memmap(
        embedding_file,
        mode="w+",
        dtype="float32",
        shape=(total, dimension)
    )
    embeddings.flush()

    checkpoint_file.write_text("0")
    start = 0
    print("Starting embedding generation from the beginning.")

else:
    raise FileNotFoundError(
        "Only one output file exists. Check the embedding and checkpoint "
        "files before proceeding; do not overwrite them blindly."
    )

# 4. Generate embeddings in batches
for start_idx in range(start, total, batch_size):
    end_idx = min(start_idx + batch_size, total)

    batch_vectors = model.encode(
        patent_documents[start_idx:end_idx],
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings[start_idx:end_idx] = batch_vectors
    embeddings.flush()

    # Save progress only after the batch is written
    checkpoint_file.write_text(str(end_idx))

    print(f"Saved {end_idx}/{total} patent embeddings")

# 5. Finish
embeddings.flush()
del embeddings

print("\nEmbedding generation completed!")
print("Embedding file:", embedding_file.resolve())
print("Total documents:", total)
print("Vector dimensions:", dimension)


NameError: name 'patent_documents' is not defined

In [2]:

# Create patent documents from the cleaned dataset

def create_patent_document(row):
    return (
        f"Application Number: {row['Application Number']}\n"
        f"Title: {row['Title']}\n"
        f"Field of Invention: {row['Field Of Invention']}\n"
        f"IPC Classification: {row['Classification (IPC)']}\n"
        f"Inventor: {row['Inventor Name']}\n"
        f"Applicant: {row['Applicant Name']}\n"
        f"Status: {row['Status']}\n"
        f"Application Filing Date: {row['Application Filing Date']}\n"
        f"Publication Number: {row['Publication Number']}"
    )

patent_documents = deduplicated_df.apply(
    create_patent_document, axis=1
).tolist()

print("Total patent documents:", len(patent_documents))
print("First document:\n", patent_documents[0])


NameError: name 'deduplicated_df' is not defined

In [3]:
print("patent_documents:", "patent_documents" in globals())
print("deduplicated_df:", "deduplicated_df" in globals())
print("model:", "model" in globals())

patent_documents: False
deduplicated_df: False
model: False


In [4]:
from pathlib import Path

for folder in ["../data/processed", "../data/raw"]:
    path = Path(folder)
    print(f"\n{path.resolve()}")

    if path.exists():
        for file in path.iterdir():
            print(file.name)
    else:
        print("Folder does not exist")


E:\Indian-patent\indian-patent-kg-rag\data\processed

E:\Indian-patent\indian-patent-kg-rag\data\raw
2011.xlsx


In [5]:

import pandas as pd
from pathlib import Path

# Load the existing dataset
df = pd.read_excel("../data/raw/2011.xlsx")

# Select the same columns used earlier
selected_columns = [
    "Application Number",
    "Title",
    "Field Of Invention",
    "Classification (IPC)",
    "Inventor Name",
    "Applicant Name",
    "Status",
    "Application Filing Date",
    "Publication Number"
]

clean_df = df[selected_columns].copy()

text_columns = [
    "Title", "Field Of Invention", "Classification (IPC)",
    "Inventor Name", "Applicant Name", "Status",
    "Application Number", "Publication Number"
]

clean_df[text_columns] = clean_df[text_columns].fillna("")

# Remove exact duplicates only
deduplicated_df = clean_df.drop_duplicates().copy()

# Recreate patent documents
def create_patent_document(row):
    return (
        f"Application Number: {row['Application Number']}\n"
        f"Title: {row['Title']}\n"
        f"Field of Invention: {row['Field Of Invention']}\n"
        f"IPC Classification: {row['Classification (IPC)']}\n"
        f"Inventor: {row['Inventor Name']}\n"
        f"Applicant: {row['Applicant Name']}\n"
        f"Status: {row['Status']}\n"
        f"Application Filing Date: {row['Application Filing Date']}\n"
        f"Publication Number: {row['Publication Number']}"
    )

patent_documents = deduplicated_df.apply(
    create_patent_document, axis=1
).tolist()

print("Dataset shape:", df.shape)
print("Patent documents:", len(patent_documents))
print("First document:\n", patent_documents[0])


Dataset shape: (39820, 35)
Patent documents: 38726
First document:
 Application Number: 1/MUM/2011
Title: WATER LIFTING SYSTEM
Field of Invention: MECHANICAL ENGINEERING
IPC Classification: F04F3/00; F04F1/00
Inventor: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Applicant: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Status: Published
Application Filing Date: 01/01/2011
Publication Number: 26/2013


In [6]:

from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print("Model loaded successfully!")
print("Embedding dimensions:", model.get_embedding_dimension())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Embedding dimensions: 384


In [7]:

from pathlib import Path
import numpy as np

# Output files
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

embedding_file = output_dir / "patent_embeddings.npy"
checkpoint_file = output_dir / "patent_embeddings_progress.txt"

# Configuration
total = len(patent_documents)
dimension = model.get_embedding_dimension()
batch_size = 32

print("Total documents:", total)
print("Embedding dimensions:", dimension)

# Resume previous work if both files exist
if embedding_file.exists() and checkpoint_file.exists():
    embeddings = np.load(embedding_file, mmap_mode="r+")

    if embeddings.shape != (total, dimension):
        raise ValueError("Embedding file shape mismatch.")

    start = int(checkpoint_file.read_text().strip())

    if not 0 <= start <= total or (
        start != total and start % batch_size != 0
    ):
        raise ValueError("Invalid checkpoint value.")

    print(f"Resuming from document {start}/{total}")

# Start fresh if neither file exists
elif not embedding_file.exists() and not checkpoint_file.exists():
    embeddings = np.lib.format.open_memmap(
        embedding_file,
        mode="w+",
        dtype="float32",
        shape=(total, dimension)
    )
    embeddings.flush()
    checkpoint_file.write_text("0")
    start = 0
    print("Starting from the beginning.")

else:
    raise FileNotFoundError(
        "Only one output file exists. Check both files before proceeding."
    )

# Generate and save each batch
for start_idx in range(start, total, batch_size):
    end_idx = min(start_idx + batch_size, total)

    batch_vectors = model.encode(
        patent_documents[start_idx:end_idx],
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings[start_idx:end_idx] = batch_vectors
    embeddings.flush()

    # Update checkpoint after saving the batch
    checkpoint_file.write_text(str(end_idx))

    print(f"Saved {end_idx}/{total} documents")

embeddings.flush()
del embeddings

print("\nAll embeddings generated successfully!")
print("Embedding file:", embedding_file.resolve())
print("Expected shape:", (total, dimension))


Total documents: 38726
Embedding dimensions: 384
Starting from the beginning.
Saved 32/38726 documents
Saved 64/38726 documents
Saved 96/38726 documents
Saved 128/38726 documents
Saved 160/38726 documents
Saved 192/38726 documents
Saved 224/38726 documents
Saved 256/38726 documents
Saved 288/38726 documents
Saved 320/38726 documents
Saved 352/38726 documents
Saved 384/38726 documents
Saved 416/38726 documents
Saved 448/38726 documents
Saved 480/38726 documents
Saved 512/38726 documents
Saved 544/38726 documents
Saved 576/38726 documents
Saved 608/38726 documents
Saved 640/38726 documents
Saved 672/38726 documents
Saved 704/38726 documents
Saved 736/38726 documents
Saved 768/38726 documents
Saved 800/38726 documents
Saved 832/38726 documents
Saved 864/38726 documents
Saved 896/38726 documents
Saved 928/38726 documents
Saved 960/38726 documents
Saved 992/38726 documents
Saved 1024/38726 documents
Saved 1056/38726 documents
Saved 1088/38726 documents
Saved 1120/38726 documents
Saved 1152/

KeyboardInterrupt: 

In [8]:
from pathlib import Path

checkpoint_file = Path("../data/processed/patent_embeddings_progress.txt")

print("Last saved checkpoint:", checkpoint_file.read_text().strip())

Last saved checkpoint: 5024


In [9]:
from pathlib import Path
import numpy as np

embedding_file = Path("../data/processed/patent_embeddings.npy")
checkpoint_file = Path("../data/processed/patent_embeddings_progress.txt")

embeddings = np.load(embedding_file, mmap_mode="r")

print("Embedding shape:", embeddings.shape)
print("Checkpoint:", checkpoint_file.read_text().strip())
print("Data type:", embeddings.dtype)
print("First vector sample:", embeddings[0][:5])
print("All values finite:", np.isfinite(embeddings).all())

Embedding shape: (38726, 384)
Checkpoint: 5024
Data type: float32
First vector sample: [-0.13783747  0.03175641 -0.01981623 -0.06288746 -0.01794686]
All values finite: True


In [10]:

import faiss
import numpy as np
from pathlib import Path

# File paths
output_dir = Path("../data/processed")
embedding_file = output_dir / "patent_embeddings.npy"
checkpoint_file = output_dir / "patent_embeddings_progress.txt"
index_file = output_dir / "patent_faiss.index"

# Read the number of completed embeddings
saved_count = int(checkpoint_file.read_text().strip())

# Load only completed embeddings
embeddings = np.load(embedding_file, mmap_mode="r")
valid_embeddings = np.asarray(
    embeddings[:saved_count],
    dtype=np.float32
)

# Create the FAISS index
dimension = valid_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add normalized vectors for cosine-similarity search
index.add(valid_embeddings)

# Save the index
faiss.write_index(index, str(index_file))

print("FAISS index created successfully!")
print("Indexed patents:", index.ntotal)
print("Vector dimensions:", dimension)
print("Index saved at:", index_file.resolve())


FAISS index created successfully!
Indexed patents: 5024
Vector dimensions: 384
Index saved at: E:\Indian-patent\indian-patent-kg-rag\data\processed\patent_faiss.index


In [11]:

# Search for patents related to water lifting systems

query = "water lifting system"

query_vector = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_vector, 5)

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]), start=1
):
    patent = deduplicated_df.iloc[int(idx)]

    print(f"\n--- Result {rank} ---")
    print(f"Similarity score: {score:.4f}")
    print("Application Number:", patent["Application Number"])
    print("Title:", patent["Title"])
    print("Field:", patent["Field Of Invention"])
    print("IPC:", patent["Classification (IPC)"])



--- Result 1 ---
Similarity score: 0.5320
Application Number: 114/KOLNP/2011
Title: A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER
Field: MECHANICAL ENGINEERING
IPC: B63B 21/50,F03B 17/06

--- Result 2 ---
Similarity score: 0.4998
Application Number: 1/MUM/2011
Title: WATER LIFTING SYSTEM
Field: MECHANICAL ENGINEERING
IPC: F04F3/00; F04F1/00

--- Result 3 ---
Similarity score: 0.4557
Application Number: 80/MUMNP/2011
Title: WATER CARRYING LIFT
Field: MECHANICAL ENGINEERING
IPC: F04B 19/08

--- Result 4 ---
Similarity score: 0.4492
Application Number: 555/KOLNP/2011
Title: WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED
Field: MECHANICAL ENGINEERING
IPC: F03B 13/18

--- Result 5 ---
Similarity score: 0.4420
Application Number: 449/CHE/2011
Title: A SYSTEM FOR GENERATING HYDRO-ELECTRIC POWER FROM WATER RESOURCE SUCH AS OVERHEAD WATER TANK OR WATER DELIVERY PIPE OF AGRICULTURAL PUMP
Field: MECHANICAL ENGINEERING
IPC: f03b


In [1]:
import pickle
from pathlib import Path

graph_path = Path("../data/processed/patent_knowledge_graph.gpickle")

with open(graph_path, "rb") as f:
    G = pickle.load(f)

print("Knowledge Graph loaded successfully!")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Knowledge Graph loaded successfully!
Nodes: 157820
Edges: 289117


In [3]:
patent_node = "PATENT:1/MUM/2011"

print("Patent exists:", patent_node in G)

if patent_node in G:
    print("\nOutgoing connections:")
    for _, target, data in G.out_edges(patent_node, data=True):
        print("Target:", target, "| Relationship:", data)

    print("\nIncoming connections:")
    for source, _, data in G.in_edges(patent_node, data=True):
        print("Source:", source, "| Relationship:", data)

Patent exists: True

Outgoing connections:
Target: INVENTOR:Koladiya Bhikhubhai Devjibhai | Relationship: {'relationship': 'INVENTED_BY'}
Target: INVENTOR:Vaghasiya Mansukhbhai Madhabhai | Relationship: {'relationship': 'INVENTED_BY'}
Target: APPLICANT:Koladiya Bhikhubhai Devjibhai | Relationship: {'relationship': 'FILED_BY'}
Target: APPLICANT:Vaghasiya Mansukhbhai Madhabhai | Relationship: {'relationship': 'FILED_BY'}
Target: IPC:F04F3/00 | Relationship: {'relationship': 'CLASSIFIED_AS'}
Target: IPC:F04F1/00 | Relationship: {'relationship': 'CLASSIFIED_AS'}
Target: STATUS:Published | Relationship: {'relationship': 'HAS_STATUS'}
Target: FIELD:MECHANICAL ENGINEERING | Relationship: {'relationship': 'FIELD_OF'}

Incoming connections:


In [4]:
for ipc_node in ["IPC:F04F3/00", "IPC:F04F1/00"]:
    print(f"\nIPC node: {ipc_node}")
    print("Node exists:", ipc_node in G)

    if ipc_node in G:
        print("Incoming connections:")
        for source, _, data in G.in_edges(ipc_node, data=True):
            print(source, data)

        print("Outgoing connections:")
        for _, target, data in G.out_edges(ipc_node, data=True):
            print(target, data)


IPC node: IPC:F04F3/00
Node exists: True
Incoming connections:
PATENT:1/MUM/2011 {'relationship': 'CLASSIFIED_AS'}
Outgoing connections:

IPC node: IPC:F04F1/00
Node exists: True
Incoming connections:
PATENT:1/MUM/2011 {'relationship': 'CLASSIFIED_AS'}
Outgoing connections:


In [5]:
ipc_edges = [
    (source, target)
    for source, target, data in G.edges(data=True)
    if data.get("relationship") == "CLASSIFIED_AS"
]

print("Total CLASSIFIED_AS edges:", len(ipc_edges))
print("Sample IPC edges:")
for source, target in ipc_edges[:10]:
    print(source, "->", target)

Total CLASSIFIED_AS edges: 46814
Sample IPC edges:
PATENT:1/MUM/2011 -> IPC:F04F3/00
PATENT:1/MUM/2011 -> IPC:F04F1/00
PATENT:1/KOLNP/2011 -> IPC:H04L 12/18
PATENT:1/KOLNP/2011 -> IPC:G06Q 30/00
PATENT:2/KOLNP/2011 -> IPC:C02F 1/00
PATENT:2395/MUMNP/2012 -> IPC:G03B19/00
PATENT:2395/MUMNP/2012 -> IPC:G03B43/00
PATENT:2395/MUMNP/2012 -> IPC:G03B17/00
PATENT:49/DELNP/2011 -> IPC:A61B 8/00
PATENT:2394/MUMNP/2012 -> IPC:H04N13/00


In [6]:
for ipc_node in ["IPC:F04F3/00", "IPC:F04F1/00"]:
    print(f"\nIPC: {ipc_node}")

    patents = [
        node
        for node in G.predecessors(ipc_node)
        if node.startswith("PATENT:")
    ]

    print("Connected patents:", len(patents))
    print("Sample patents:", patents[:10])


IPC: IPC:F04F3/00
Connected patents: 1
Sample patents: ['PATENT:1/MUM/2011']

IPC: IPC:F04F1/00
Connected patents: 1
Sample patents: ['PATENT:1/MUM/2011']


In [7]:
import pandas as pd
import re

df_check = pd.read_excel(
    "../data/raw/2011.xlsx",
    usecols=["Application Number", "Title", "Classification (IPC)"]
)

# Find original records containing F04F
mask = df_check["Classification (IPC)"].astype(str).str.contains(
    "F04F", case=False, na=False
)

print("Records containing F04F:", mask.sum())

print(
    df_check.loc[
        mask,
        ["Application Number", "Classification (IPC)"]
    ].head(15).to_string(index=False)
)

# Normalize IPC codes by removing spaces and converting to uppercase
def normalize_ipc(value):
    if pd.isna(value):
        return []
    codes = re.split(r"[;,]", str(value))
    return [
        re.sub(r"\s+", "", code).upper()
        for code in codes
        if code.strip()
    ]

df_check["normalized_ipc"] = df_check["Classification (IPC)"].apply(
    normalize_ipc
)

for target in ["F04F3/00", "F04F1/00"]:
    matches = df_check[
        df_check["normalized_ipc"].apply(lambda codes: target in codes)
    ]
    print(f"\n{target}: {len(matches)} matching records")
    print(
        matches[["Application Number", "Classification (IPC)"]]
        .head(10)
        .to_string(index=False)
    )

Records containing F04F: 7
Application Number Classification (IPC)
        1/MUM/2011   F04F3/00; F04F1/00
       39/KOL/2011             F04F5/24
   1364/DELNP/2011           F04F 13/00
   2637/CHENP/2011   F04F7/00, F04F5/10
   4457/DELNP/2011            F04F 5/16
   9174/DELNP/2011            F04F 1/18
   9176/DELNP/2011            F04F 1/18

F04F3/00: 1 matching records
Application Number Classification (IPC)
        1/MUM/2011   F04F3/00; F04F1/00

F04F1/00: 1 matching records
Application Number Classification (IPC)
        1/MUM/2011   F04F3/00; F04F1/00


In [8]:
from collections import Counter

ipc_counts = Counter(
    ipc
    for codes in df_check["normalized_ipc"]
    for ipc in codes
)

print("Top 10 IPC classifications:\n")

for ipc, count in ipc_counts.most_common(10):
    print(f"{ipc}: {count} records")

Top 10 IPC classifications:

B23B: 399 records
A61K: 395 records
C07D: 344 records
G06F: 242 records
H04L: 205 records
H04L29/06: 197 records
C07C: 184 records
G06F17/30: 182 records
C12Q1/68: 166 records
G06Q: 153 records


In [9]:
ipc_node = "IPC:B23B"

print("IPC node exists:", ipc_node in G)

if ipc_node in G:
    patents = [
        node.replace("PATENT:", "")
        for node in G.predecessors(ipc_node)
        if node.startswith("PATENT:")
    ]

    print("Patents connected to B23B:", len(patents))
    print("Sample patents:", patents[:10])

IPC node exists: True
Patents connected to B23B: 393
Sample patents: ['127/CHE/2011', '387/DEL/2011', '931/DEL/2011', '938/DEL/2011', '977/DEL/2011', '1001/DEL/2011', '1005/DEL/2011', '1008/DEL/2011', '1028/DEL/2011', '1060/DEL/2011']


In [10]:
def get_related_patents(application_number):
    patent_node = f"PATENT:{application_number}"

    if patent_node not in G:
        return []

    related_patents = set()

    # Find IPC classifications for this patent
    for _, ipc_node, edge_data in G.out_edges(
        patent_node, data=True
    ):
        if edge_data.get("relationship") != "CLASSIFIED_AS":
            continue

        # Find all patents sharing each IPC classification
        for other_patent in G.predecessors(ipc_node):
            if (
                other_patent.startswith("PATENT:")
                and other_patent != patent_node
            ):
                related_patents.add(
                    other_patent.replace("PATENT:", "")
                )

    return sorted(related_patents)


# Test with a patent using the common B23B classification
print("Related patents:", len(get_related_patents("127/CHE/2011")))
print("Sample:", get_related_patents("127/CHE/2011")[:10])

Related patents: 392
Sample: ['1001/DEL/2011', '10049/DELNP/2011', '1005/DEL/2011', '1008/DEL/2011', '10232/DELNP/2011', '10235/DELNP/2011', '10236/DELNP/2011', '10255/DELNP/2011', '10256/DELNP/2011', '1028/DEL/2011']


In [12]:
from pathlib import Path

checkpoint_file = Path("../data/processed/patent_embeddings_progress.txt")
embedding_file = Path("../data/processed/patent_embeddings.npy")

print("Embedding file exists:", embedding_file.exists())
print("Checkpoint file exists:", checkpoint_file.exists())

if checkpoint_file.exists():
    completed = int(checkpoint_file.read_text().strip())
    total = 38726

    print(f"Completed: {completed}/{total}")
    print(f"Remaining: {total - completed}")
    print(f"Progress: {completed / total * 100:.2f}%")

Embedding file exists: True
Checkpoint file exists: True
Completed: 5024/38726
Remaining: 33702
Progress: 12.97%


In [13]:
import pandas as pd

df = pd.read_excel("../data/raw/2011.xlsx")

selected_columns = [
    "Application Number",
    "Title",
    "Field Of Invention",
    "Classification (IPC)",
    "Inventor Name",
    "Applicant Name",
    "Status",
    "Application Filing Date",
    "Publication Number"
]

clean_df = df[selected_columns].copy()

text_columns = [
    "Title", "Field Of Invention", "Classification (IPC)",
    "Inventor Name", "Applicant Name", "Status",
    "Application Number", "Publication Number"
]

clean_df[text_columns] = clean_df[text_columns].fillna("")
deduplicated_df = clean_df.drop_duplicates().copy()

def create_patent_document(row):
    return (
        f"Application Number: {row['Application Number']}\n"
        f"Title: {row['Title']}\n"
        f"Field of Invention: {row['Field Of Invention']}\n"
        f"IPC Classification: {row['Classification (IPC)']}\n"
        f"Inventor: {row['Inventor Name']}\n"
        f"Applicant: {row['Applicant Name']}\n"
        f"Status: {row['Status']}\n"
        f"Application Filing Date: {row['Application Filing Date']}\n"
        f"Publication Number: {row['Publication Number']}"
    )

patent_documents = deduplicated_df.apply(
    create_patent_document, axis=1
).tolist()

print("Patent documents:", len(patent_documents))
print("First document:\n", patent_documents[0])

Patent documents: 38726
First document:
 Application Number: 1/MUM/2011
Title: WATER LIFTING SYSTEM
Field of Invention: MECHANICAL ENGINEERING
IPC Classification: F04F3/00; F04F1/00
Inventor: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Applicant: #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai
Status: Published
Application Filing Date: 01/01/2011
Publication Number: 26/2013


In [14]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print("Embedding dimension:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


In [15]:
from pathlib import Path
import numpy as np

output_dir = Path("../data/processed")
embedding_file = output_dir / "patent_embeddings.npy"
checkpoint_file = output_dir / "patent_embeddings_progress.txt"

total = len(patent_documents)
dimension = model.get_embedding_dimension()
batch_size = 32

embeddings = np.load(embedding_file, mmap_mode="r+")

start = int(checkpoint_file.read_text().strip())

print(f"Resuming from {start}/{total}")

for start_idx in range(start, total, batch_size):
    end_idx = min(start_idx + batch_size, total)

    batch_vectors = model.encode(
        patent_documents[start_idx:end_idx],
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings[start_idx:end_idx] = batch_vectors
    embeddings.flush()
    checkpoint_file.write_text(str(end_idx))

    if end_idx % 320 == 0 or end_idx == total:
        print(f"Saved {end_idx}/{total} documents")

embeddings.flush()
del embeddings

print("All embeddings generated successfully!")

Resuming from 5024/38726
Saved 5120/38726 documents
Saved 5440/38726 documents
Saved 5760/38726 documents
Saved 6080/38726 documents
Saved 6400/38726 documents
Saved 6720/38726 documents
Saved 7040/38726 documents
Saved 7360/38726 documents
Saved 7680/38726 documents
Saved 8000/38726 documents
Saved 8320/38726 documents
Saved 8640/38726 documents
Saved 8960/38726 documents
Saved 9280/38726 documents
Saved 9600/38726 documents
Saved 9920/38726 documents
Saved 10240/38726 documents


KeyboardInterrupt: 

In [16]:
from pathlib import Path

checkpoint_file = Path("../data/processed/patent_embeddings_progress.txt")

completed = int(checkpoint_file.read_text().strip())

print("Completed embeddings:", completed)
print("Remaining to reach 10,000:", max(0, 10000 - completed))

Completed embeddings: 10304
Remaining to reach 10,000: 0


In [17]:
import faiss
import numpy as np
from pathlib import Path

output_dir = Path("../data/processed")

embedding_file = output_dir / "patent_embeddings.npy"
checkpoint_file = output_dir / "patent_embeddings_progress.txt"
index_file = output_dir / "patent_faiss.index"

# Read the confirmed checkpoint
completed = int(checkpoint_file.read_text().strip())

# Load only completed embeddings
embeddings = np.load(embedding_file, mmap_mode="r")
valid_embeddings = np.asarray(
    embeddings[:completed],
    dtype=np.float32
)

# Build FAISS index
index = faiss.IndexFlatIP(valid_embeddings.shape[1])
index.add(valid_embeddings)

# Save index
faiss.write_index(index, str(index_file))

print("FAISS index created successfully!")
print("Vectors indexed:", index.ntotal)
print("Dimensions:", index.d)
print("Saved to:", index_file.resolve())

FAISS index created successfully!
Vectors indexed: 10304
Dimensions: 384
Saved to: E:\Indian-patent\indian-patent-kg-rag\data\processed\patent_faiss.index


In [18]:
import faiss
import numpy as np

# Load the saved FAISS index
index = faiss.read_index(
    "../data/processed/patent_faiss.index"
)

# Search query
query = "water lifting system"

# Convert query into an embedding
query_vector = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Retrieve the top 5 matching patents
scores, indices = index.search(query_vector, 5)

print("Query:", query)
print("\nTop matching patents:\n")

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]), start=1
):
    patent = deduplicated_df.iloc[int(idx)]

    print(f"{rank}. {patent['Title']}")
    print(f"   Application Number: {patent['Application Number']}")
    print(f"   Similarity Score: {score:.4f}\n")

Query: water lifting system

Top matching patents:

1. A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER
   Application Number: 114/KOLNP/2011
   Similarity Score: 0.5320

2. WATER LIFTING SYSTEM
   Application Number: 1/MUM/2011
   Similarity Score: 0.4998

3. WATER CARRYING LIFT
   Application Number: 80/MUMNP/2011
   Similarity Score: 0.4557

4. WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED
   Application Number: 555/KOLNP/2011
   Similarity Score: 0.4492

5. METHOD AND SYSTEM FOR PUNCTUAL FASTENING A WATERPROOFING MEMBRANE TO HYDRAULIC WORKS
   Application Number: 470/MUMNP/2011
   Similarity Score: 0.4460



In [19]:
query_patent = "1/MUM/2011"

related = get_related_patents(query_patent)

print("Query patent:", query_patent)
print("Related patents found:", len(related))
print("\nFirst 10 related application numbers:")

for patent_id in related[:10]:
    print(patent_id)

Query patent: 1/MUM/2011
Related patents found: 0

First 10 related application numbers:


In [20]:
query_patent = "127/CHE/2011"

related_patents = get_related_patents(query_patent)

print("Query patent:", query_patent)
print("Related patents found:", len(related_patents))
print("\nFirst 10 related patents:")

for patent_id in related_patents[:10]:
    print(patent_id)

Query patent: 127/CHE/2011
Related patents found: 392

First 10 related patents:
1001/DEL/2011
10049/DELNP/2011
1005/DEL/2011
1008/DEL/2011
10232/DELNP/2011
10235/DELNP/2011
10236/DELNP/2011
10255/DELNP/2011
10256/DELNP/2011
1028/DEL/2011


## 6. Hybrid Search (FAISS + Knowledge Graph)

Combines semantic search using FAISS with graph-based patent retrieval using NetworkX to find relevant and related Indian patents.

In [21]:
def hybrid_patent_search(query, top_k=5, graph_limit=20):
    # 1. Semantic search using FAISS
    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_vector, top_k)

    results = {}
    graph_seeds = []

    for idx, score in zip(indices[0], scores[0]):
        if idx < 0:
            continue

        patent = deduplicated_df.iloc[int(idx)]
        app_no = str(patent["Application Number"]).strip()

        if not app_no:
            continue

        results[app_no] = {
            "application_number": app_no,
            "title": patent["Title"],
            "score": float(score),
            "source": "FAISS"
        }

        graph_seeds.append(app_no)

    # 2. Expand results using Knowledge Graph relationships
    for app_no in graph_seeds:
        related = get_related_patents(app_no)

        for related_id in related[:graph_limit]:
            if related_id in results:
                continue

            matching_rows = deduplicated_df[
                deduplicated_df["Application Number"].astype(str).str.strip()
                == related_id
            ]

            if matching_rows.empty:
                continue

            patent = matching_rows.iloc[0]

            results[related_id] = {
                "application_number": related_id,
                "title": patent["Title"],
                "score": None,
                "source": f"Knowledge Graph (related to {app_no})"
            }

    return list(results.values())

In [22]:
results = hybrid_patent_search(
    "water lifting system",
    top_k=5,
    graph_limit=20
)

print("Total combined patents:", len(results))

for rank, patent in enumerate(results[:15], start=1):
    print(f"\n{rank}. {patent['title']}")
    print("   Application:", patent["application_number"])
    print("   Source:", patent["source"])

    if patent["score"] is not None:
        print(f"   FAISS similarity: {patent['score']:.4f}")

Total combined patents: 20

1. A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER
   Application: 114/KOLNP/2011
   Source: FAISS
   FAISS similarity: 0.5320

2. WATER LIFTING SYSTEM
   Application: 1/MUM/2011
   Source: FAISS
   FAISS similarity: 0.4998

3. WATER CARRYING LIFT
   Application: 80/MUMNP/2011
   Source: FAISS
   FAISS similarity: 0.4557

4. WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED
   Application: 555/KOLNP/2011
   Source: FAISS
   FAISS similarity: 0.4492

5. METHOD AND SYSTEM FOR PUNCTUAL FASTENING A WATERPROOFING MEMBRANE TO HYDRAULIC WORKS
   Application: 470/MUMNP/2011
   Source: FAISS
   FAISS similarity: 0.4460

6. "WATER POWER GENERATORS"
   Application: 10216/DELNP/2011
   Source: Knowledge Graph (related to 114/KOLNP/2011)

7. FLUID DYNAMIC MACHINE WITH BLADE ROTORS
   Application: 405/KOLNP/2011
   Source: Knowledge Graph (related to 114/KOLNP/2011)

8. "THE HYDROELECTRIC SYSTEM SEAWATER."
   Applicati

## 7. Evidence-Based Answer Generation

Uses the retrieved patent metadata from hybrid search to generate answers grounded in available evidence, including application numbers, patent titles, retrieval sources, and similarity scores where available.

In [23]:

def generate_patent_answer(query, top_k=5, graph_limit=20):
    results = hybrid_patent_search(
        query,
        top_k=top_k,
        graph_limit=graph_limit
    )

    if not results:
        return {
            "question": query,
            "answer": "No matching patents were found in the available dataset.",
            "evidence": []
        }

    evidence = []

    for patent in results:
        evidence.append({
            "application_number": patent["application_number"],
            "title": patent["title"],
            "source": patent["source"],
            "similarity_score": patent["score"]
        })

    answer_lines = [
        f"Question: {query}",
        "",
        "Retrieved patent evidence:",
    ]

    for i, patent in enumerate(evidence, start=1):
        answer_lines.append(
            f"{i}. {patent['title']}\n"
            f"   Application Number: {patent['application_number']}\n"
            f"   Retrieval Source: {patent['source']}"
        )

        if patent["similarity_score"] is not None:
            answer_lines.append(
                f"   FAISS Similarity: "
                f"{patent['similarity_score']:.4f}"
            )

    answer_lines.extend([
        "",
        "Limitation: These results are based on patent metadata "
        "and shared IPC classifications. They do not establish "
        "technical equivalence or legally confirmed prior art."
    ])

    return {
        "question": query,
        "answer": "\n".join(answer_lines),
        "evidence": evidence
    }


In [24]:

response = generate_patent_answer("water lifting system")

print(response["answer"])


Question: water lifting system

Retrieved patent evidence:
1. A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER
   Application Number: 114/KOLNP/2011
   Retrieval Source: FAISS
   FAISS Similarity: 0.5320
2. WATER LIFTING SYSTEM
   Application Number: 1/MUM/2011
   Retrieval Source: FAISS
   FAISS Similarity: 0.4998
3. WATER CARRYING LIFT
   Application Number: 80/MUMNP/2011
   Retrieval Source: FAISS
   FAISS Similarity: 0.4557
4. WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED
   Application Number: 555/KOLNP/2011
   Retrieval Source: FAISS
   FAISS Similarity: 0.4492
5. METHOD AND SYSTEM FOR PUNCTUAL FASTENING A WATERPROOFING MEMBRANE TO HYDRAULIC WORKS
   Application Number: 470/MUMNP/2011
   Retrieval Source: FAISS
   FAISS Similarity: 0.4460
6. "WATER POWER GENERATORS"
   Application Number: 10216/DELNP/2011
   Retrieval Source: Knowledge Graph (related to 114/KOLNP/2011)
7. FLUID DYNAMIC MACHINE WITH BLADE ROTORS
   Applicati

## 8. LLM-Based Answer Generation

Uses a Large Language Model (LLM) to summarize hybrid retrieval results into a natural-language answer, grounded in retrieved patent metadata and supported by application-number references.

## 8. LLM-Based Answer Generation Using Groq

Integrates the Groq API with the hybrid retrieval pipeline to generate natural-language answers grounded in retrieved patent metadata and supporting evidence.

In [25]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import groq
print("Groq library installed successfully!")

Groq library installed successfully!


In [30]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

key = os.getenv("GROQ_API_KEY")

print("API key loaded:", bool(key))
print("Key format looks valid:", bool(key and key.startswith("gsk_")))

API key loaded: True
Key format looks valid: True


In [31]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv("../.env", override=True)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Reply only: Groq is working!"}
    ],
    max_completion_tokens=50
)

print(response.choices[0].message.content)

In [32]:

from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv("../.env", override=True)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Sending request to Groq...")

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    max_completion_tokens=100
)

print("Response received!")
print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", repr(response.choices[0].message.content))


Sending request to Groq...
Response received!
Finish reason: stop
Answer: 'Hello!'


## 9. LLM-Based Patent Question Answering

Integrates Groq with hybrid retrieval using FAISS and NetworkX to generate natural-language answers grounded in retrieved patent metadata, with application-number references and explicit limitations.

In [33]:

def ask_patent_question(query, top_k=5, graph_limit=20):
    patents = hybrid_patent_search(
        query,
        top_k=top_k,
        graph_limit=graph_limit
    )

    if not patents:
        return "No matching patents were found in the dataset."

    evidence = []

    for patent in patents:
        evidence.append({
            "application_number": patent["application_number"],
            "title": patent["title"],
            "source": patent["source"],
            "similarity_score": patent["score"]
        })

    prompt = f"""
You are an assistant for searching Indian patent metadata.

Answer the user's question using ONLY the patent evidence below.

Rules:
- Mention relevant patent titles and application numbers.
- Do not invent technical details absent from the evidence.
- Treat shared IPC classifications as related search results,
  not proof that patents describe the same invention.
- State when the available evidence is insufficient.
- Do not claim legal prior-art conclusions.

User question:
{query}

Retrieved patent evidence:
{evidence}

Give a concise, clear answer with supporting application numbers.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "Answer accurately and cite the supplied patent application numbers."
            },
            {"role": "user", "content": prompt}
        ],
        max_completion_tokens=700
    )

    return response.choices[0].message.content


In [34]:

answer = ask_patent_question("water lifting system")
print(answer)


**Indian patent applications that directly refer to a “water‑lifting” concept**

| Application No. | Title |
|------------------|-------|
| **1/MUM/2011** | *Water Lifting System* |
| **80/MUMNP/2011** | *Water Carrying Lift* (related concept) |

The application **1/MUM/2011** is the only patent that uses the exact phrase “Water Lifting System.”  The second application, **80/MUMNP/2011**, describes a “Water Carrying Lift,” which is a closely related type of water‑handling apparatus.

*No further technical details are provided in the supplied evidence, so any discussion of the specific mechanics, design features, or intended use must be sought from the full patent documents themselves.*


## 10. Patent Evidence Validation

Validates generated answers against the available patent dataset by checking application numbers and titles, helping ensure that cited patent records exist in the source data.

In [35]:

def validate_patent_evidence(patents):
    validated = []

    for patent in patents:
        app_no = str(patent["application_number"]).strip()

        matches = deduplicated_df[
            deduplicated_df["Application Number"]
            .astype(str).str.strip() == app_no
        ]

        if matches.empty:
            validated.append({
                "application_number": app_no,
                "valid": False,
                "message": "Application number not found"
            })
        else:
            validated.append({
                "application_number": app_no,
                "valid": True,
                "title": str(matches.iloc[0]["Title"])
            })

    return validated


# Test using hybrid search results
test_patents = hybrid_patent_search("water lifting system")
validation_results = validate_patent_evidence(test_patents)

for item in validation_results:
    print(item)


{'application_number': '114/KOLNP/2011', 'valid': True, 'title': 'A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER'}
{'application_number': '1/MUM/2011', 'valid': True, 'title': 'WATER LIFTING SYSTEM'}
{'application_number': '80/MUMNP/2011', 'valid': True, 'title': 'WATER CARRYING LIFT'}
{'application_number': '555/KOLNP/2011', 'valid': True, 'title': 'WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED'}
{'application_number': '470/MUMNP/2011', 'valid': True, 'title': 'METHOD AND SYSTEM FOR PUNCTUAL FASTENING A WATERPROOFING MEMBRANE TO HYDRAULIC WORKS'}
{'application_number': '10216/DELNP/2011', 'valid': True, 'title': '"WATER POWER GENERATORS"'}
{'application_number': '405/KOLNP/2011', 'valid': True, 'title': 'FLUID DYNAMIC MACHINE WITH BLADE ROTORS'}
{'application_number': '5270/DELNP/2011', 'valid': True, 'title': '"THE HYDROELECTRIC SYSTEM SEAWATER."'}
{'application_number': '7718/DELNP/2011', 'valid': True, 'title': 'IN-PIPE HYD

## 11. Exact Patent Title Search

Searches the patent dataset for titles containing a user-provided phrase and returns the matching application numbers and titles.

In [36]:

def exact_title_search(query):
    matches = deduplicated_df[
        deduplicated_df["Title"]
        .fillna("")
        .str.contains(query, case=False, regex=False)
    ]

    return matches[
        ["Application Number", "Title"]
    ].drop_duplicates()


# Test
exact_matches = exact_title_search("water lifting system")

print("Exact phrase matches:", len(exact_matches))
print(exact_matches.to_string(index=False))


Exact phrase matches: 1
Application Number                Title
        1/MUM/2011 WATER LIFTING SYSTEM


## 12. Integrated Hybrid Patent Retrieval

Combines exact title matching, FAISS semantic retrieval, and NetworkX knowledge-graph retrieval to identify relevant patent records and provide evidence for LLM-based question answering.

In [37]:

def integrated_patent_search(query, top_k=5, graph_limit=20):
    results = {}

    # 1. Exact title search
    exact_matches = exact_title_search(query)

    for _, row in exact_matches.iterrows():
        app_no = str(row["Application Number"]).strip()

        if app_no:
            results[app_no] = {
                "application_number": app_no,
                "title": row["Title"],
                "source": ["Exact Title Search"],
                "score": None
            }

    # 2. FAISS + Knowledge Graph retrieval
    hybrid_results = hybrid_patent_search(
        query,
        top_k=top_k,
        graph_limit=graph_limit
    )

    for patent in hybrid_results:
        app_no = str(patent["application_number"]).strip()

        if not app_no:
            continue

        if app_no in results:
            results[app_no]["source"].append(patent["source"])

            if patent["score"] is not None:
                results[app_no]["score"] = patent["score"]
        else:
            results[app_no] = {
                "application_number": app_no,
                "title": patent["title"],
                "source": [patent["source"]],
                "score": patent["score"]
            }

    return list(results.values())


In [38]:

results = integrated_patent_search("water lifting system")

print("Total unique patents:", len(results))

for i, patent in enumerate(results[:10], start=1):
    print(f"\n{i}. {patent['title']}")
    print("Application:", patent["application_number"])
    print("Source:", ", ".join(patent["source"]))

    if patent["score"] is not None:
        print(f"FAISS similarity: {patent['score']:.4f}")


Total unique patents: 20

1. WATER LIFTING SYSTEM
Application: 1/MUM/2011
Source: Exact Title Search, FAISS
FAISS similarity: 0.4998

2. A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER
Application: 114/KOLNP/2011
Source: FAISS
FAISS similarity: 0.5320

3. WATER CARRYING LIFT
Application: 80/MUMNP/2011
Source: FAISS
FAISS similarity: 0.4557

4. WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED
Application: 555/KOLNP/2011
Source: FAISS
FAISS similarity: 0.4492

5. METHOD AND SYSTEM FOR PUNCTUAL FASTENING A WATERPROOFING MEMBRANE TO HYDRAULIC WORKS
Application: 470/MUMNP/2011
Source: FAISS
FAISS similarity: 0.4460

6. "WATER POWER GENERATORS"
Application: 10216/DELNP/2011
Source: Knowledge Graph (related to 114/KOLNP/2011)

7. FLUID DYNAMIC MACHINE WITH BLADE ROTORS
Application: 405/KOLNP/2011
Source: Knowledge Graph (related to 114/KOLNP/2011)

8. "THE HYDROELECTRIC SYSTEM SEAWATER."
Application: 5270/DELNP/2011
Source: Knowledge Grap

## 13. Integrated Patent Question Answering with Groq

Generates natural-language answers using exact title matches, FAISS semantic search, and NetworkX knowledge-graph results. Answers are grounded in retrieved patent metadata and include application-number references.

In [51]:
def ask_integrated_patent_question(query, top_k=5, graph_limit=20):
    if not query or not query.strip():
        return "Please enter a patent-related query."

    patents = integrated_patent_search(
        query,
        top_k=top_k,
        graph_limit=graph_limit
    )

    if not patents:
        return "No matching patents were found in the dataset."

    evidence = [
        {
            "application_number": p["application_number"],
            "title": p["title"],
            "source": p["source"],
            "similarity_score": p["score"]
        }
        for p in patents
    ]

    prompt = f"""
You are an Indian patent metadata search assistant.

Answer the user's question using only the supplied evidence.

Rules:
- Cite relevant patent application numbers and titles.
- Distinguish exact title matches from semantic matches.
- Explain that graph results share IPC classifications.
- Do not invent technical details.
- Do not claim that a patent is legally confirmed prior art.
- Mention limitations when full patent descriptions or claims
  are unavailable.

Question:
{query}

Retrieved evidence:
{evidence}

Write a clear, concise answer with supporting application numbers.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "Provide evidence-grounded patent search answers."
            },
            {"role": "user", "content": prompt}
        ],
        max_completion_tokens=700
    )

    return response.choices[0].message.content

In [40]:

answer = ask_integrated_patent_question("water lifting system")
print(answer)


**Patents related to a “water lifting system”**

| Application | Title | Source | Notes |
|-------------|-------|--------|-------|
| **1/MUM/2011** | **“WATER LIFTING SYSTEM”** | Exact title search (FAISS) | Exact match – the title itself is identical to the keyword. |
| **80/MUMNP/2011** | **“WATER CARRYING LIFT”** | FAISS (semantic similarity 0.46) | Semantic match – the title is close in meaning but not identical. |

---

### Additional, more broadly related patents

The search also returned a cluster of patents linked through the **Knowledge Graph** to the 2011 “Mooring‑plant” application (114/KOLNP/2011). These patents share IPC classifications that involve marine or fluid‑dynamic energy systems but do **not** describe a conventional water‑lifting device. They are listed for reference only:

| Application | Title (excerpt) | Source |
|-------------|-----------------|--------|
| 10216/DELNP/2011 | “WATER POWER GENERATORS” | Knowledge Graph |
| 405/KOLNP/2011 | “FLUID DYNAMIC MACHIN

In [41]:
answer = ask_integrated_patent_question("water lifting system")
print(answer)

**Water‑lifting systems – patent‑search results**

| Match type | Application number | Title | Source |
|------------|--------------------|-------|--------|
| **Exact title match** | **1/MUM/2011** | *WATER LIFTING SYSTEM* | Exact Title Search / FAISS |
| **Semantic similarity** | 80/MUMNP/2011 | *WATER CARRYING LIFT* | FAISS (similarity ≈ 0.46) |
| | 114/KOLNP/2011 | *A SYSTEM FOR MOORING A FLOATING PLANT FOR THE PRODUCTION OF ENERGY FROM CURRENTS IN WATER* | FAISS (similarity ≈ 0.53) |
| | 555/KOLNP/2011 | *WAVE ACTUATED PUMP AND MEANS OF CONNECTING SAME TO THE SEABED* | FAISS (similarity ≈ 0.45) |

> **Graph‑derived links**  
> The Knowledge‑Graph entries (e.g., 10216/DELNP/2011 “WATER POWER GENERATORS”, 405/KOLNP/2011 “FLUID DYNAMIC MACHINE WITH BLADE ROTORS”, etc.) are shown because they share IPC classifications with 114/KOLNP/2011. Graph results do **not** indicate a direct title or claim overlap; they simply suggest related technical themes.

---

### What this tells you

* **E

Agriculture Patent Search Results

The integrated patent retrieval system was tested using the query “agricultural machinery”.

In [42]:
query = "agricultural machinery"

results = integrated_patent_search(
    query,
    top_k=5,
    graph_limit=20
)

for patent in results[:10]:
    print("Application:", patent["application_number"])
    print("Title:", patent["title"])
    print("Source:", ", ".join(patent["source"]))
    print("Score:", patent["score"])
    print("-" * 50)

Application: 946/DEL/2011
Title: "A SYSTEM FOR MEASUREMENT OF VERTICAL ANGLE OF LOWER LINK OF TRACTOR FOR DIFFERENT FARMING OPERATIONS"
Source: FAISS
Score: 0.5007277727127075
--------------------------------------------------
Application: 902/KOLNP/2011
Title: AGRICULTURAL TRACTOR
Source: FAISS
Score: 0.4984389543533325
--------------------------------------------------
Application: 234/CHE/2011
Title: A MACHINE FOR PELLETING OF SEEDS
Source: FAISS
Score: 0.48507454991340637
--------------------------------------------------
Application: 572/CHE/2011
Title: IMPROVED DEVICE FOR ACHIEVING UNIFORMITY IN FIRST STEAMING (RAW PADDY STEAMING) IN PARBOILING OF GRAINS
Source: FAISS
Score: 0.47482895851135254
--------------------------------------------------
Application: 992/CHE/2011
Title: AGRICULTURAL DRIVING MECHANISM AND RELATED TOOL
Source: FAISS
Score: 0.4679127335548401
--------------------------------------------------
Application: 2199/DEL/2011
Title: "AUTOMATED DETECTION OF CROP ANOM

FINAL TESTING

In [43]:
exact_title_search("WATER LIFTING SYSTEM")

,Application Number,Title
0,1/MUM/2011,WATER LIFTING SYSTEM


In [44]:
results = integrated_patent_search("agricultural machinery")

print("Total results:", len(results))

for patent in results[:5]:
    print("Application:", patent["application_number"])
    print("Title:", patent["title"])
    print("Source:", patent["source"])
    print("-" * 40)

Total results: 36
Application: 946/DEL/2011
Title: "A SYSTEM FOR MEASUREMENT OF VERTICAL ANGLE OF LOWER LINK OF TRACTOR FOR DIFFERENT FARMING OPERATIONS"
Source: ['FAISS']
----------------------------------------
Application: 902/KOLNP/2011
Title: AGRICULTURAL TRACTOR
Source: ['FAISS']
----------------------------------------
Application: 234/CHE/2011
Title: A MACHINE FOR PELLETING OF SEEDS
Source: ['FAISS']
----------------------------------------
Application: 572/CHE/2011
Title: IMPROVED DEVICE FOR ACHIEVING UNIFORMITY IN FIRST STEAMING (RAW PADDY STEAMING) IN PARBOILING OF GRAINS
Source: ['FAISS']
----------------------------------------
Application: 992/CHE/2011
Title: AGRICULTURAL DRIVING MECHANISM AND RELATED TOOL
Source: ['FAISS']
----------------------------------------


In [45]:
answer = ask_integrated_patent_question(
    "agricultural machinery"
)
print(answer)

**Agricultural‑machinery‑related patent applications (Indian filings, 2011)**  

| Application No. | Title | Match type |
|------------------|-------|------------|
| **902/KOLNP/2011** | *AGRICULTURAL TRACTOR* | **Exact title match** |
| **992/CHE/2011** | *AGRICULTURAL DRIVING MECHANISM AND RELATED TOOL* | **Exact title match** |
| 946/DEL/2011 | *A SYSTEM FOR MEASUREMENT OF VERTICAL ANGLE OF LOWER LINK OF TRACTOR FOR DIFFERENT FARMING OPERATIONS* | Semantic match – the title contains key terms “tractor” and “farming operations” that are relevant to agricultural machinery |
| 234/CHE/2011 | *A MACHINE FOR PELLETING OF SEEDS* | Semantic match – the device is a seed‑pelleting machine, a common piece of agricultural equipment |
| 992/CHE/2011 (knowledge‑graph link) | *TILLING MACHINE* | Semantic match – the graph associates it with the same IPC family as the 946/DEL/2011 application |
| 992/CHE/2011 (knowledge‑graph link) | *MOVING LEVER HAND WEEDER* | Semantic match |
| 992/CHE/2011 (kn

In [46]:
for patent in results:
    if "Knowledge Graph" in str(patent["source"]):
        print(
            patent["application_number"],
            "|",
            patent["title"],
            "|",
            patent["source"]
        )

2199/DEL/2011 | "AUTOMATED DETECTION OF CROP ANOMALIES" | ['Knowledge Graph (related to 946/DEL/2011)']
2521/CHE/2011 | TILLING MACHINE | ['Knowledge Graph (related to 946/DEL/2011)']
3748/CHE/2011 | MOVING LEVER HAND WEEDER | ['Knowledge Graph (related to 946/DEL/2011)']
3924/CHE/2011 | RAMLING WEEDING MACHINES | ['Knowledge Graph (related to 946/DEL/2011)']
4164/CHE/2011 | URVARA - ADJUSTABLE AGRICULTURAL SPRAYS | ['Knowledge Graph (related to 946/DEL/2011)']
741/DEL/2011 | "A ROOTER STRAND FOR AIR LAYERING". | ['Knowledge Graph (related to 946/DEL/2011)']
2328/MUMNP/2011 | PLUNGING CROSS-TRACK CONSTANT VELOCITY JOINT | ['Knowledge Graph (related to 902/KOLNP/2011)']
245/MUMNP/2011 | DYNAMIC DAMPER | ['Knowledge Graph (related to 902/KOLNP/2011)']
233/CHE/2011 | A SEED PELLETING MIXTURE COMPOSITION AND SEED PELLETING PROTOCOL | ['Knowledge Graph (related to 234/CHE/2011)']
3812/CHE/2011 | DESIGNER SEED TAPE | ['Knowledge Graph (related to 234/CHE/2011)']
128/DEL/2011 | "A PROCESS FOR

In [47]:
answer = ask_integrated_patent_question(
    "Find patents related to seed pelleting and explain their relevance."
)

print(answer)

**Patents that directly address seed pelleting**

| Application No. | Title | Relevance |
|------------------|-------|-----------|
| **233/CHE/2011** | *A SEED PELLETING MIXTURE COMPOSITION AND SEED PELLETING PROTOCOL* | **Exact‑title match.**  The application discloses a specific mixture (ingredients, proportions, and processing conditions) that can be applied to seeds to form pellets, as well as a step‑by‑step protocol for pelleting.  This makes it the most directly relevant prior art on seed pelleting. |
| **234/CHE/2011** | *A MACHINE FOR PELLETING OF SEEDS* | **Exact‑title match.**  This filing describes an apparatus (and likely associated control logic) designed to physically pelletize seeds.  It complements the composition from 233/CHE/2011 by providing the hardware needed for the process. |

**Patents that are semantically related to seed pelleting**

| Application No. | Title | Why it’s semantically related |
|------------------|-------|------------------------------|
| **521/

In [57]:
answer = ask_integrated_patent_question("system")
print(answer)

BadRequestError: Error code: 400 - {'error': {'message': 'Please reduce the length of the messages or completion.', 'type': 'invalid_request_error', 'param': 'messages'}}

In [58]:
from pathlib import Path

processed = Path("../data/processed")

for filename in [
    "patent_embeddings.npy",
    "patent_embeddings_progress.txt",
    "patent_faiss.index",
    "patent_knowledge_graph.gpickle",
]:
    path = processed / filename
    print(filename, "EXISTS" if path.exists() else "MISSING")

patent_embeddings.npy EXISTS
patent_embeddings_progress.txt EXISTS
patent_faiss.index EXISTS
patent_knowledge_graph.gpickle EXISTS
